### Staff Directory Scraper (`HofUniversityStaffScraper`)

#### Overview

Scrapes staff contact and profile data from the Hof University staff directory
using **Selenium** (JavaScript-rendered page) with department-by-department filtering.

- **Source URL:** `https://www.hof-university.com/about-hof-university/university-organization/staff-directory.html`
- **Tool:** Selenium + Chrome WebDriver (headless)
- **Output:** `hof_university_staff.json`

---

#### Covered Departments

| # | Department |
|---|-----------|
| 1 | Business Department |
| 2 | Center for Languages and Intercultural Competence |
| 3 | Department of Computer Science |
| 4 | Department of Interdisciplinary and Innovative Sciences |
| 5 | Engineering Department |
| 6 | Graduate School |

---



In [2]:
import requests
from bs4 import BeautifulSoup
import json
import time
import os
import json
from typing import Dict
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import re
from urllib.parse import urljoin
import unicodedata

class HofUniversityStaffScraper:
    def __init__(self, headless=True):
        """Initialize the scraper with Chrome WebDriver"""
        self.base_url = "https://www.hof-university.com/about-hof-university/university-organization/staff-directory.html"
        self.driver = None
        self.headless = headless
        self.staff_data = []
        
        # Known departments
        self.departments = [
            {"value": "Business Department", "text": "Business Department"},
            {"value": "Center for Languages and Intercultural Competence", "text": "Center for Languages and Intercultural Competence"},
            {"value": "Department of Computer Science", "text": "Department of Computer Science"},
            {"value": "Department of Interdisciplinary and Innovative Sciences", "text": "Department of Interdisciplinary and Innovative Sciences"},
            {"value": "Engineering Department", "text": "Engineering Department"},
            {"value": "Graduate School", "text": "Graduate School"}
        ]
        
    def setup_driver(self):
        """Setup Chrome WebDriver"""
        chrome_options = Options()
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        
        if self.headless:
            chrome_options.add_argument("--headless")
        
        try:
            self.driver = webdriver.Chrome(options=chrome_options)
            self.driver.set_page_load_timeout(30)
            return True
        except Exception as e:
            print(f"Error setting up Chrome driver: {e}")
            return False
    
    def normalize_text(self, text):
        """Normalize text for better searchability"""
        if not text:
            return ""
        
        # Remove accents and normalize unicode
        text = unicodedata.normalize('NFKD', text)
        text = ''.join(c for c in text if not unicodedata.combining(c))
        
        # Convert to lowercase and remove extra whitespace
        text = re.sub(r'\s+', ' ', text.lower().strip())
        
        # Remove common punctuation but keep useful chars
        text = re.sub(r'[^\w\s@.-]', ' ', text)
        
        return text
    
    def generate_professor_id(self, full_name):
        """Generate a unique identifier for the professor"""
        if not full_name:
            return None
        
        # Extract just the name part (remove titles)
        name_without_title = re.sub(r'^(Prof\.?\s+)?(Dr\.?\s+)?', '', full_name, flags=re.IGNORECASE)
        
        # Convert to lowercase, replace spaces with underscores
        prof_id = self.normalize_text(name_without_title)
        prof_id = re.sub(r'\s+', '_', prof_id)
        prof_id = re.sub(r'[^\w_]', '', prof_id)
        
        return prof_id
    
    def extract_names(self, full_name):
        """Extract different name components and variations"""
        if not full_name:
            return {}, []
        
        # Extract title (Prof., Dr., Prof. Dr.)
        title_match = re.match(r'^((?:Prof\.?\s+)?(?:Dr\.?\s+)?)', full_name, re.IGNORECASE)
        title = title_match.group(1).strip() if title_match else ""
        
        # Extract name without title
        name_without_title = re.sub(r'^(Prof\.?\s+)?(Dr\.?\s+)?', '', full_name, flags=re.IGNORECASE).strip()
        
        # Split into first and last names
        name_parts = name_without_title.split()
        first_name = name_parts[0] if name_parts else ""
        last_name = name_parts[-1] if len(name_parts) > 1 else ""
        
        # Generate name variations
        name_variations = []
        
        if name_without_title:
            name_variations.append(name_without_title)
        
        if last_name:
            name_variations.append(last_name)
        
        if title and name_without_title:
            if "Prof" in title:
                name_variations.extend([
                    f"Prof. {name_without_title}",
                    f"Professor {name_without_title}",
                    f"Professor {last_name}",
                ])
            
            if "Dr" in title:
                name_variations.extend([
                    f"Dr. {name_without_title}",
                    f"Doctor {name_without_title}",
                ])
            
            name_variations.append(full_name)
        
        # Remove duplicates while preserving order
        name_variations = list(dict.fromkeys(name_variations))
        
        name_components = {
            "first_name": first_name,
            "last_name": last_name, 
            "title": title
        }
        
        return name_components, name_variations
    
    def parse_contact_info(self, phone_string, email_string):
        """Parse and structure contact information"""
        contact = {
            "email": email_string or "",
            "phone_office": "",
            "phone_mobile": "",
            "phone_all": phone_string or ""
        }
        
        if phone_string:
            # Split multiple phone numbers
            phones = [p.strip() for p in phone_string.split('|') if p.strip()]
            
            if len(phones) >= 1:
                contact["phone_office"] = phones[0]
            if len(phones) >= 2:
                contact["phone_mobile"] = phones[1]
        
        return contact
    
    def parse_location_info(self, office_string, campus="", building="", room=""):
        """Parse and structure location information"""
        location = {
            "campus": campus.strip(),
            "building": building.strip(),
            "room": room.strip()
        }
        
        # Try to extract from office string if individual components not provided
        if office_string and not (campus or building or room):
            campus_match = re.search(r'campus:\s*([^,]+)', office_string, re.IGNORECASE)
            building_match = re.search(r'building:\s*([^,]+)', office_string, re.IGNORECASE) 
            room_match = re.search(r'room\s+number:\s*([^,]+)', office_string, re.IGNORECASE)
            
            if campus_match:
                location["campus"] = campus_match.group(1).strip()
            if building_match:
                location["building"] = building_match.group(1).strip()
            if room_match:
                location["room"] = room_match.group(1).strip()
        
        return location
    
    def extract_staff_info(self, element):
        """Extract individual staff member information from a person element"""
        # Raw data storage
        raw_data = {
            'name': '',
            'position': '',
            'email': '',
            'phone': '',
            'department': '',
            'teaching_areas': '',
            'campus': '',
            'building': '',
            'room': ''
        }
        
        try:
            # Extract name from the header link
            name_selectors = [
                'a[data-bs-target*="mitarbeiterModal"]',
                '.header-meta a',
                'a.open-modal',
                '.header-meta .header', 
                '.header-meta h1, .header-meta h2, .header-meta h3, .header-meta h4',
                'a.header'
            ]
            
            for selector in name_selectors:
                try:
                    name_elements = element.find_elements(By.CSS_SELECTOR, selector)
                    for name_element in name_elements:
                        name_text = name_element.text.strip()
                        if name_text and len(name_text) > 3:
                            raw_data['name'] = name_text
                            break
                    if raw_data['name']:
                        break
                except:
                    continue
            
            # Alternative: Extract name from img title
            if not raw_data['name']:
                try:
                    img_element = element.find_element(By.TAG_NAME, 'img')
                    img_title = img_element.get_attribute('title')
                    if img_title and '|' in img_title:
                        name_part = img_title.split('|')[0].strip()
                        if name_part:
                            raw_data['name'] = name_part
                except:
                    pass
            
            # Extract from header-meta if still no name
            if not raw_data['name']:
                try:
                    header_meta = element.find_element(By.CSS_SELECTOR, '.header-meta')
                    all_text = header_meta.get_attribute('textContent').strip()
                    lines = [line.strip() for line in all_text.split('\n') if line.strip()]
                    if lines:
                        raw_data['name'] = lines[0]
                except:
                    pass
            
            # Extract positions from header-meta paragraphs
            try:
                positions = []
                header_meta = element.find_element(By.CSS_SELECTOR, '.header-meta')
                position_paragraphs = header_meta.find_elements(By.TAG_NAME, 'p')
                
                for p in position_paragraphs:
                    text = p.get_attribute('textContent').strip()
                    if text and text not in positions:
                        positions.append(text)
                
                raw_data['position'] = ' | '.join(positions)
            except:
                pass
            
            # Extract contact details
            try:
                meta_selectors = ['.meta > div', '.meta div', 'div:has(.category-text)']
                meta_divs = []
                
                for selector in meta_selectors:
                    try:
                        meta_divs = element.find_elements(By.CSS_SELECTOR, selector)
                        if meta_divs:
                            break
                    except:
                        continue
                
                if not meta_divs:
                    all_divs = element.find_elements(By.TAG_NAME, 'div')
                    for div in all_divs:
                        if any(keyword in div.text.lower() for keyword in ['tel:', 'phone', 'email', '@', 'contact', 'room', 'building']):
                            meta_divs.append(div)
                
                for div in meta_divs:
                    try:
                        div_text_content = div.get_attribute('textContent')
                        div_text = div_text_content.lower() if div_text_content else ""
                        
                        # Look for contact information
                        if 'contact' in div_text or 'tel:' in div_text or '@' in div_text or 'phone' in div_text:
                            # Extract phone numbers
                            phone_links = div.find_elements(By.CSS_SELECTOR, 'a[href^="tel:"]')
                            phones = []
                            for phone_link in phone_links:
                                phone = phone_link.get_attribute('textContent').strip()
                                if phone:
                                    phones.append(phone)
                            if phones:
                                raw_data['phone'] = ' | '.join(phones)
                            
                            # Extract email
                            email_patterns = [
                                'a[data-mailto-token]',
                                'a[href^="mailto:"]'
                            ]
                            
                            for pattern in email_patterns:
                                try:
                                    email_elements = div.find_elements(By.CSS_SELECTOR, pattern)
                                    for email_element in email_elements:
                                        email_text = email_element.get_attribute('textContent').strip()
                                        if email_text:
                                            if '(at)' in email_text:
                                                raw_data['email'] = email_text.replace('(at)', '@')
                                            else:
                                                raw_data['email'] = email_text
                                            break
                                    if raw_data['email']:
                                        break
                                except:
                                    continue
                            
                            # Search for email in text if not found
                            if not raw_data['email'] and div_text_content:
                                email_match = re.search(r'([a-zA-Z0-9._%+-]+\(at\)[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})', div_text_content)
                                if email_match:
                                    raw_data['email'] = email_match.group(1).replace('(at)', '@')
                        
                        # Look for location information
                        if 'location' in div_text or 'campus' in div_text or 'building' in div_text or 'room' in div_text:
                            location_text = div_text_content or ""
                            
                            campus_match = re.search(r'Campus\s+([^\|]+)', location_text, re.IGNORECASE)
                            if campus_match:
                                raw_data['campus'] = campus_match.group(1).strip()
                            
                            building_match = re.search(r'Building\s+([^\|]+)', location_text, re.IGNORECASE)
                            if building_match:
                                raw_data['building'] = building_match.group(1).strip()
                            
                            room_match = re.search(r'Room\s+([^\|]+)', location_text, re.IGNORECASE)
                            if room_match:
                                raw_data['room'] = room_match.group(1).strip()
                        
                        # Look for teaching information
                        if 'teaching' in div_text or 'subject' in div_text:
                            try:
                                teaching_elements = div.find_elements(By.TAG_NAME, 'p')
                                for p in teaching_elements:
                                    text = p.get_attribute('textContent').strip()
                                    if text and 'teaching' not in text.lower() and len(text) > 5:
                                        raw_data['teaching_areas'] = text
                                        break
                            except:
                                pass
                    except:
                        continue
                        
            except:
                pass
            
            # Validate - only proceed if we have meaningful data
            if not (raw_data['name'] or raw_data['email'] or raw_data['phone']):
                return None
            
            # Convert to structured format
            professor_id = self.generate_professor_id(raw_data['name'])
            name_components, name_variations = self.extract_names(raw_data['name'])
            
            # Parse contact information
            contact = self.parse_contact_info(raw_data['phone'], raw_data['email'])
            
            # Build office string
            office_parts = []
            if raw_data['room']:
                office_parts.append(f"Room number: {raw_data['room']}")
            if raw_data['building']:
                office_parts.append(f"building: {raw_data['building']}")
            if raw_data['campus']:
                office_parts.append(f"campus: {raw_data['campus']}")
            office_string = " , ".join(office_parts) if office_parts else ""
            
            # Parse location information
            location = self.parse_location_info(office_string, raw_data['campus'], raw_data['building'], raw_data['room'])
            
            # Create structured staff info
            staff_info = {
                "professor_id": professor_id,
                "full_name": raw_data['name'],
                "first_name": name_components['first_name'],
                "last_name": name_components['last_name'], 
                "title": name_components['title'],
                "name_variations": name_variations,
                "position": raw_data['position'],
                "contact": contact,
                "office": office_string,
                "location": location,
                "department": raw_data['department'],
                "teaching_areas": raw_data['teaching_areas']
            }
            
            return staff_info
            
        except Exception as e:
            return None
    
    def scrape_department_staff(self, department):
        """Scrape staff data for a specific department"""
        try:
            print(f"Scraping department: {department['text']}")
            
            # Navigate to page
            self.driver.get(self.base_url)
            
            # Wait for page to load
            WebDriverWait(self.driver, 15).until(
                lambda driver: driver.execute_script("return document.readyState") == "complete"
            )
            time.sleep(3)
            
            # Select department
            try:
                dropdown = self.driver.find_element(By.ID, "fakultsuche")
                select = Select(dropdown)
                select.select_by_value(department['value'])
                time.sleep(5)
            except Exception as e:
                print(f"Error selecting department: {e}")
                return []
            
            # Find all person elements
            person_elements = self.driver.find_elements(By.CSS_SELECTOR, '.person')
            print(f"Found {len(person_elements)} staff elements")
            
            department_staff = []
            
            for i, element in enumerate(person_elements):
                try:
                    staff_info = self.extract_staff_info(element)
                    
                    if staff_info:
                        staff_info['department'] = department['text']
                        department_staff.append(staff_info)
                        print(f"  {i+1}. {staff_info['full_name']} (ID: {staff_info['professor_id']})")
                    else:
                        print(f"  {i+1}. Failed to extract data")
                        
                except Exception as e:
                    print(f"  {i+1}. Error processing element: {e}")
                    continue
            
            print(f"Successfully extracted {len(department_staff)} staff members")
            return department_staff
            
        except Exception as e:
            print(f"Error scraping department {department['text']}: {e}")
            return []
    
    def scrape_all_departments(self):
        """Scrape staff data from all departments"""
        if not self.setup_driver():
            print("Failed to setup WebDriver")
            return False
        
        try:
            all_staff = []
            
            for dept in self.departments:
                try:
                    staff_data = self.scrape_department_staff(dept)
                    all_staff.extend(staff_data)
                    time.sleep(3)
                    
                except Exception as e:
                    print(f"Error scraping department {dept['text']}: {e}")
                    continue
            
            self.staff_data = all_staff
            print(f"Total staff members extracted: {len(all_staff)}")
            return True
            
        finally:
            if self.driver:
                self.driver.quit()
    
    def save_to_json(self, filename="data/hof_university_staff.json"):
        """Save staff data to JSON file"""
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(self.staff_data, f, indent=2, ensure_ascii=False)
            print(f"Data saved to {filename}")
        except Exception as e:
            print(f"Error saving to JSON: {e}")
    
    def display_summary(self):
        """Display a summary of scraped data"""
        if not self.staff_data:
            print("No data to display")
            return
        
        # Group by department
        dept_counts = {}
        for staff in self.staff_data:
            dept = staff.get('department', 'Unknown')
            dept_counts[dept] = dept_counts.get(dept, 0) + 1
        
        print(f"\nScraping Summary:")
        print(f"Total staff members: {len(self.staff_data)}")
        print(f"Total departments: {len(dept_counts)}")
        
        print("\nStaff by department:")
        for dept, count in sorted(dept_counts.items()):
            print(f"  {dept}: {count} members")
        
        # Data completeness
        with_names = sum(1 for s in self.staff_data if s.get('full_name'))
        with_emails = sum(1 for s in self.staff_data if s.get('contact', {}).get('email'))
        with_phones = sum(1 for s in self.staff_data if s.get('contact', {}).get('phone_all'))
        with_positions = sum(1 for s in self.staff_data if s.get('position'))
        
        print(f"\nData completeness:")
        print(f"  Names: {with_names}/{len(self.staff_data)} ({with_names/len(self.staff_data)*100:.1f}%)")
        print(f"  Emails: {with_emails}/{len(self.staff_data)} ({with_emails/len(self.staff_data)*100:.1f}%)")
        print(f"  Phones: {with_phones}/{len(self.staff_data)} ({with_phones/len(self.staff_data)*100:.1f}%)")
        print(f"  Positions: {with_positions}/{len(self.staff_data)} ({with_positions/len(self.staff_data)*100:.1f}%)")

def main():
    """Main function to run the scraper"""
    print("Hof University Staff Directory Scraper")
    print("=" * 50)
    
    # Create scraper
    scraper = HofUniversityStaffScraper(headless=True)

    os.makedirs("data", exist_ok=True)

    
    try:
        # Scrape all departments
        success = scraper.scrape_all_departments()
        
        if success and scraper.staff_data:
            # Display summary
            scraper.display_summary()
            
            # Save data
            scraper.save_to_json()
            
            print(f"\nScraping completed successfully!")
            print(f"Data saved to hof_university_staff.json")
            
        else:
            print("Scraping failed or no data found")
            
    except KeyboardInterrupt:
        print("\nOperation cancelled by user")
    except Exception as e:
        print(f"Unexpected error: {e}")

if __name__ == "__main__":
    main()

Hof University Staff Directory Scraper
Scraping department: Business Department
Found 42 staff elements
  1. Prof. Dr. Jens Kirchner (ID: jens_kirchner)
  2. Prof. Dr. Christine Brautsch (ID: christine_brautsch)
  3. Prof. Dr. Astrid Nöfer (ID: astrid_nofer)
  4. Prof. Dr. Dunja Stadtmann (ID: dunja_stadtmann)
  5. Prof. Dr. Viktoria Bachmann (ID: viktoria_bachmann)
  6. Prof. Dr. Thomas Meuche (ID: thomas_meuche)
  7. Prof. Dr. Joachim Riedl (ID: joachim_riedl)
  8. Prof. Dr. Andreas Wagener (ID: andreas_wagener)
  9. Prof. Dr. Beatrix Weber (ID: beatrix_weber)
  10. Prof. Dr. Stefan Wengler (ID: stefan_wengler)
  11. Claudia Bilski (ID: claudia_bilski)
  12. Prof. Dr. Dietmar Boerner (ID: dietmar_boerner)
  13. Prof. Dr. Ulrich Entrup (ID: ulrich_entrup)
  14. Prof. Dr. Markus Finn (ID: markus_finn)
  15. Prof. Dr.-Ing. Roland Fischer (ID: dring_roland_fischer)
  16. Prof. Dr. Miriam Frey-Knoll (ID: miriam_freyknoll)
  17. Prof. Dr. Michaela Gebele (ID: michaela_gebele)
  18. Prof. D